# 06 — Real Hardware, Data Saving & REST Service

This notebook covers the three things you do differently on real hardware vs simulation:

1. Connect to the QICK board via Pyro4
2. Populate `system_cfg.py` with your lab's channel map
3. Save / reload experimental data (HDF5)
4. Start the REST service for remote experiment submission

> **Note:** cells in section 1 require a live QICK board and will not run in simulation.

## 1. Connecting to the QICK board

`BaseExperiment.connect_pyro4()` opens the Pyro4 proxy and activates the hardware session used by experiments.


In [ ]:
# HARDWARE ONLY
import sys; sys.path.insert(0, '../')
from QickworkspaceV2 import BaseExperiment

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82', ns_port=8888, proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)
print('Connected:', BaseExperiment._session_name)


## 2. Hardware config ? edit `system_cfg.py`

Open `QickworkspaceV2/config/system_cfg.py` and maintain the actual channel map and
starting parameters there. `config_list` is the hardware source of truth. Important
current keys include `nqz_res`, `ro_length`, `res_sigma`, `res_gain_ge`, `nqz_qb`,
`qb_mixer`, `qb_gain_ge`, `sigma_ge`, `trig_time`, and `relax_delay`.

The data path is supplied when connecting:

```python
BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82', ns_port=8888, proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)
```

Load the maintained config with:


In [ ]:
from QickworkspaceV2 import ExperimentConfig
from QickworkspaceV2.config.system_cfg import config_list   # your lab config

cfg_all = ExperimentConfig(config_list)
cfg = cfg_all.get_qubit('Q1')
print('Loaded qubits:', cfg_all.qubit_names())

## 3. Run an experiment and save data

`ExperimentData.save(path)` writes an HDF5 file with raw IQ, axes, and fit params.
Use `cfg_all.to_yaml(q_id='Q1')` when you need a formatted configuration.

In [ ]:
# Real hardware resonator smoke test
from qick.asm_v2 import QickSweep1D
from QickworkspaceV2.experiments.resonator import ResonatorSpec

center = cfg_all.get_qubit('Q1')['res_freq_ge']
run_cfg = cfg_all.get_qubit('Q1')
run_cfg.update([
    ('steps', 51),
    ('res_freq_ge', QickSweep1D('freqloop', center-10, center+10)),
    ('relax_delay', 0),
])

expt = ResonatorSpec(run_cfg)
result = expt.run(py_avg=5)


In [ ]:
from pathlib import Path
from QickworkspaceV2 import ExperimentData

# Native standalone HDF5
save_path = result.save(data_root=Path(r'D:\Labber_Data\Jay\test'))
print('Saved to:', save_path)

reloaded = ExperimentData.load(save_path)
print('Reloaded experiment_type:', reloaded.experiment_type)
print('Reloaded scalar_result:', reloaded.scalar_result)


In [ ]:
# Combined Labber + /metagroup HDF5 (random-ID filename by default)
labber_path = expt.saveLabber('Q1', config_all=cfg_all)
print('Labber hybrid file:', labber_path)

# Use filename_mode='sequential' for _001, _002, ... names:
# expt.saveLabber('Q1', config_all=cfg_all, filename_mode='sequential')


## 4. REST service status

The REST service is experimental. The current module exposes health, schema, experiment-job,
and calibration endpoints, but completed experiment jobs still reference the missing
`QickworkspaceV2.data.serializer` module. Use the health/schema endpoints for inspection;
do not rely on remote experiment result serialization yet.


In [ ]:
from fastapi.testclient import TestClient
from QickworkspaceV2.service import create_app

app = create_app(config_all=cfg_all)
client = TestClient(app)
print(client.get('/health').json())


In [ ]:
schema = client.get('/experiments/schema').json()
print('Registered experiments:', len(schema['experiments']))
for item in schema['experiments'][:8]:
    print(item['id'], '->', item['class_path'])


In [ ]:
# Current endpoints as declared by the installed service module.
for path in app.openapi()['paths']:
    print(path)


## Hardware validation checklist

Run these steps in order on first hardware connection:

1. `python -c "import QickworkspaceV2; print(QickworkspaceV2.__version__)"` — package loads
2. `BaseExperiment.connect_pyro4(IP, PORT, data_path=DATA_PATH)` — Pyro4 connection succeeds
3. `ResonatorSpec` — verify resonator dip appears at the expected frequency
4. `QubitSpec` — verify qubit peak appears
5. `PowerRabi` — verify clean Rabi oscillation
6. `Ramsey` — verify detuning < 1 MHz after QubitSpec
7. `AutoCalibrate.run(skip=('spin_echo', 't1', 'ss_opt'))` — fast first-pass calibration
8. `AutoCalibrate.run()` — full calibration